Since your goal is eventually WAN/cloud-network troubleshooting and automation, I would focus on **production-style packet analysis**, not just basic sniffing.

Microsoft internal packet-capture guidance consistently emphasizes using captures to answer questions like *Did the packet leave? Did it return? Was there packet loss, retransmission, filtering, or latency?* [\[1 - Network Traces \| PowerPoint\]](https://microsoft.sharepoint.com/teams/Azure_IaaS_Technical_Support/_layouts/15/Doc.aspx?sourcedoc=%7BCB91C8FE-7CF3-49C5-92EE-E4CE9F926E5A%7D&file=1%20-%20Network%20Traces.pptx&action=edit&mobileredirect=true&DefaultItemOpen=1), [\[1 - Network Traces \| PowerPoint\]](https://microsoft.sharepoint.com/teams/Azure_IaaS_Technical_Support/_layouts/15/Doc.aspx?sourcedoc=%7B1FF5DA81-4A3F-492F-B2BA-BB6F0821F94E%7D&file=1%20-%20Network%20Traces.pptx&action=edit&mobileredirect=true&DefaultItemOpen=1)

Below are examples that teach the same workflow used by network engineers.

***

# Project 1: Top Talkers Report

Find which IPs are generating the most traffic.

```python
from scapy.all import sniff
from scapy.layers.inet import IP

ip_count = {}

def analyze(packet):

    if IP in packet:

        src = packet[IP].src

        ip_count[src] = ip_count.get(src, 0) + 1

sniff(count=100, prn=analyze)

print("\nTop Talkers")

for ip, count in sorted(
    ip_count.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(ip, count)
```

Example:

```text
192.168.1.10 45
13.107.246.40 32
8.8.8.8 15
```

***

# Project 2: DNS Troubleshooting Tool

Useful for WAN and internet troubleshooting.

```python
from scapy.all import sniff

dns_packets = sniff(
    filter="udp port 53",
    count=20
)

print("DNS Packets:", len(dns_packets))
```

Now open:

```text
google.com
microsoft.com
bing.com
```

while the capture runs.

***

# Project 3: Detect TCP Retransmissions

Save packets first:

```python
from scapy.all import sniff, wrpcap

packets = sniff(
    filter="tcp",
    count=100
)

wrpcap(
    "tcp_capture.pcap",
    packets
)
```

Open in Wireshark.

Filter:

```text
tcp.analysis.retransmission
```

This is extremely useful for packet-loss troubleshooting. Internal Wireshark references highlight retransmissions as one of the key indicators for performance and packet-loss diagnosis. [\[# Wireshar...e and tips \| External\]](https://eng.ms/cid/4c0cf652-5649-4f2c-9b8c-004306b62ede/fid/4f7fe6836cb478b11d053441ba03e8e729e8df9b5074162ced98b4f0e079dab3), [\[WireShark \| ADO Wiki (Supportability)\]](https://Supportability.visualstudio.com/AzureAD/_wiki/wikis/AzureAD/2487360)

***

# Project 4: TCP Port Statistics

```python
from scapy.all import sniff
from scapy.layers.inet import TCP

ports = {}

def analyze(packet):

    if TCP in packet:

        port = packet[TCP].dport

        ports[port] = ports.get(port, 0) + 1

sniff(
    filter="tcp",
    count=200,
    prn=analyze
)

print("\nPORT SUMMARY")

for port, count in ports.items():

    print(
        f"Port {port}: {count}"
    )
```

Output:

```text
Port 443: 150
Port 80: 20
Port 53: 12
```

***

# Project 5: Capture Only One Host

Capture traffic to Google DNS.

```python
from scapy.all import sniff

packets = sniff(
    filter="host 8.8.8.8",
    timeout=30
)

print(
    "Packets:",
    len(packets)
)
```

Generate traffic:

```bash
ping 8.8.8.8
```

***

# Project 6: Protocol Counter

This is a very common first automation exercise.

```python
from scapy.all import sniff
from scapy.layers.inet import TCP, UDP, ICMP

stats = {
    "TCP": 0,
    "UDP": 0,
    "ICMP": 0
}

def analyze(packet):

    if TCP in packet:
        stats["TCP"] += 1

    elif UDP in packet:
        stats["UDP"] += 1

    elif ICMP in packet:
        stats["ICMP"] += 1

sniff(
    count=100,
    prn=analyze
)

print(stats)
```

Example:

```text
{
'TCP':71,
'UDP':20,
'ICMP':9
}
```

***

# Project 7: Save Capture with Timestamp

Very common production practice.

```python
from scapy.all import sniff, wrpcap
from datetime import datetime

filename = (
    "capture_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".pcap"
)

packets = sniff(
    timeout=15
)

wrpcap(
    filename,
    packets
)

print(
    f"Saved {filename}"
)
```

Internal guidance also recommends unique capture names and rotating captures to preserve troubleshooting history. [\[Use Packet...soft Learn \| Learn.Microsoft.com\]](https://learn.microsoft.com/en-us/azure/firewall/packet-capture), [\[How To:: C...work trace \| ADO Wiki (Supportability)\]](https://Supportability.visualstudio.com/Fabric/_wiki/wikis/Fabric/2694699)

***

# Project 8: Live Traffic Monitor

```python
from scapy.all import sniff
from scapy.layers.inet import IP

def analyze(packet):

    if IP in packet:

        print(
            packet[IP].src,
            "->",
            packet[IP].dst
        )

sniff(
    count=50,
    prn=analyze
)
```

Output:

```text
192.168.1.15 -> 8.8.8.8
13.107.246.40 -> 192.168.1.15
```

***

# Project 9: Read PCAP and Generate Report

```python
from scapy.all import rdpcap
from scapy.layers.inet import TCP

packets = rdpcap(
    "capture.pcap"
)

tcp_count = 0

for packet in packets:

    if TCP in packet:
        tcp_count += 1

print(
    "Total Packets:",
    len(packets)
)

print(
    "TCP Packets:",
    tcp_count
)
```

This is how many network automation tools begin.

***

# Project 10: Production-Style Health Report

```python
from scapy.all import sniff
from scapy.layers.inet import IP, TCP, UDP, ICMP

report = {
    "TCP": 0,
    "UDP": 0,
    "ICMP": 0,
    "Unique Hosts": set()
}

def analyze(packet):

    if IP in packet:

        report["Unique Hosts"].add(
            packet[IP].src
        )

    if TCP in packet:
        report["TCP"] += 1

    elif UDP in packet:
        report["UDP"] += 1

    elif ICMP in packet:
        report["ICMP"] += 1

sniff(
    count=200,
    prn=analyze
)

print("\nTRAFFIC REPORT")
print("-" * 40)

print(
    "TCP:",
    report["TCP"]
)

print(
    "UDP:",
    report["UDP"]
)

print(
    "ICMP:",
    report["ICMP"]
)

print(
    "Unique Hosts:",
    len(report["Unique Hosts"])
)
```

***

### Recommended Learning Order

1. ICMP capture + ping
2. DNS capture
3. Save PCAP
4. Read PCAP
5. Protocol counter
6. Top talkers report
7. TCP port report
8. Retransmission analysis in Wireshark
9. DNS troubleshooting dashboard
10. Build a mini packet-analyzer that exports results to CSV

Those 10 projects will teach nearly all of the Scapy skills that are useful in real network operations and WAN troubleshooting.


In [1]:
print("Hellodhdethetet")

Hellodhdethetet


In [2]:
from scapy.all import sniff
from scapy.layers.inet import IP

ip_count = {}

def analyze(packet):

    if IP in packet:

        src = packet[IP].src

        ip_count[src] = ip_count.get(src, 0) + 1

sniff(count=100, prn=analyze)

print("\nTop Talkers")

for ip, count in sorted(
    ip_count.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(ip, count)


Top Talkers
10.0.9.200 65
51.5.71.12 14
51.5.71.36 8
23.219.78.204 4
40.64.145.160 4
51.5.71.27 2
52.110.2.58 1
173.194.43.138 1
52.96.164.242 1


In [3]:
from scapy.all import sniff

dns_packets = sniff(
    filter="udp port 53",
    count=20
)

print("DNS Packets:", len(dns_packets))

DNS Packets: 20


In [4]:
from scapy.all import sniff, wrpcap

packets = sniff(
    filter="tcp",
    count=100
)

wrpcap(
    "tcp_capture.pcap",
    packets
)

In [5]:
from scapy.all import sniff
from scapy.layers.inet import TCP

ports = {}

def analyze(packet):

    if TCP in packet:

        port = packet[TCP].dport

        ports[port] = ports.get(port, 0) + 1

sniff(
    filter="tcp",
    count=200,
    prn=analyze
)

print("\nPORT SUMMARY")

for port, count in ports.items():

    print(
        f"Port {port}: {count}"
    )


PORT SUMMARY
Port 56471: 1
Port 56041: 1
Port 443: 64
Port 58119: 2
Port 52858: 2
Port 62581: 3
Port 62583: 3
Port 59460: 1
Port 52058: 1
Port 52859: 2
Port 63750: 1
Port 61986: 1
Port 80: 29
Port 52860: 3
Port 52861: 3
Port 52862: 3
Port 52863: 2
Port 52864: 16
Port 32526: 2
Port 49227: 1
Port 60079: 1
Port 52777: 1
Port 7680: 6
Port 58901: 5
Port 52865: 2
Port 57075: 1
Port 52866: 16
Port 52867: 2
Port 52868: 12
Port 52869: 12
Port 52778: 1


In [6]:
from scapy.all import sniff

packets = sniff(
    filter="host 8.8.8.8",
    timeout=30
)

print(
    "Packets:",
    len(packets)
)


Packets: 188


In [7]:
from scapy.all import sniff
from scapy.layers.inet import TCP, UDP, ICMP

stats = {
    "TCP": 0,
    "UDP": 0,
    "ICMP": 0
}

def analyze(packet):

    if TCP in packet:
        stats["TCP"] += 1

    elif UDP in packet:
        stats["UDP"] += 1

    elif ICMP in packet:
        stats["ICMP"] += 1

sniff(
    count=100,
    prn=analyze
)

print(stats)

{'TCP': 3, 'UDP': 97, 'ICMP': 0}


In [8]:
from scapy.all import sniff, wrpcap
from datetime import datetime

filename = (
    "capture_"
    + datetime.now().strftime(
        "%Y%m%d_%H%M%S"
    )
    + ".pcap"
)

packets = sniff(
    timeout=15
)

wrpcap(
    filename,
    packets
)

print(
    f"Saved {filename}"
)

Saved capture_20260712_123142.pcap


In [9]:
from scapy.all import rdpcap
from scapy.layers.inet import TCP

packets = rdpcap(
    "capture.pcap"
)

tcp_count = 0

for packet in packets:

    if TCP in packet:
        tcp_count += 1

print(
    "Total Packets:",
    len(packets)
)

print(
    "TCP Packets:",
    tcp_count
)

FileNotFoundError: [Errno 2] No such file or directory: 'capture.pcap'

In [10]:
import os

print(os.getcwd())

c:\Users\v-yaalam\Downloads\vscode_Dell_17\1.Python Learning\Network Coding\wireshark_productionready


In [11]:
import os

for file in os.listdir():
    print(file)

capture_20260712_123142.pcap
tcp_capture.pcap
wire shark production ready.ipynb


In [13]:
from scapy.all import rdpcap
from scapy.layers.inet import TCP

packets = rdpcap(
    "tcp_capture.pcap"
)

tcp_count = 0

for packet in packets:

    if TCP in packet:
        tcp_count += 1

print(
    "Total Packets:",
    len(packets)
)

print(
    "TCP Packets:",
    tcp_count
)

Total Packets: 100
TCP Packets: 100


In [14]:
from scapy.all import rdpcap
from scapy.layers.inet import TCP

packets = rdpcap(
    "capture_20260712_123142.pcap"
)

tcp_count = 0

for packet in packets:

    if TCP in packet:
        tcp_count += 1

print(
    "Total Packets:",
    len(packets)
)

print(
    "TCP Packets:",
    tcp_count
)

Total Packets: 20808
TCP Packets: 2422


In [15]:
from scapy.all import rdpcap

packets = rdpcap(
    "tcp_capture.pcap"
)

print(
    "Total Packets:",
    len(packets)
)

for i, packet in enumerate(
    packets[:10],
    start=1
):

    print(
        f"Packet {i}:",
        packet.summary()
    )

Total Packets: 100
Packet 1: Ether / IP / TCP 10.0.9.200:62581 > 40.64.145.160:https PA / Raw
Packet 2: Ether / IP / TCP 10.0.9.200:62583 > 40.64.145.160:https PA / Raw
Packet 3: Ether / IP / TCP 40.64.145.160:https > 10.0.9.200:62581 A / Padding
Packet 4: Ether / IP / TCP 40.64.145.160:https > 10.0.9.200:62583 A / Padding
Packet 5: Ether / IP / TCP 10.0.9.200:52838 > 142.250.73.78:https S
Packet 6: Ether / IP / TCP 142.250.73.78:https > 10.0.9.200:52838 SA
Packet 7: Ether / IP / TCP 10.0.9.200:52838 > 142.250.73.78:https A
Packet 8: Ether / IP / TCP 10.0.9.200:52838 > 142.250.73.78:https FA
Packet 9: Ether / IP / TCP 142.250.73.78:https > 10.0.9.200:52838 FA / Padding
Packet 10: Ether / IP / TCP 10.0.9.200:52838 > 142.250.73.78:https A


In [16]:
import os

for file in os.listdir():

    if file.endswith(".pcap"):

        print(file)

capture_20260712_123142.pcap
tcp_capture.pcap


In [17]:
import os
from scapy.all import rdpcap

pcap_files = [
    file
    for file in os.listdir()
    if file.endswith(".pcap")
]

print(
    "Using:",
    pcap_files[0]
)

packets = rdpcap(
    pcap_files[0]
)

print(
    "Total Packets:",
    len(packets)
)

Using: capture_20260712_123142.pcap
Total Packets: 20808


In [18]:
from scapy.all import rdpcap
from scapy.layers.inet import TCP, UDP, ICMP

packets = rdpcap(
    "tcp_capture.pcap"
)

tcp = 0
udp = 0
icmp = 0

for packet in packets:

    if TCP in packet:
        tcp += 1

    elif UDP in packet:
        udp += 1

    elif ICMP in packet:
        icmp += 1

print("=" * 40)
print("PCAP ANALYSIS REPORT")
print("=" * 40)

print("Total Packets:", len(packets))
print("TCP Packets  :", tcp)
print("UDP Packets  :", udp)
print("ICMP Packets :", icmp)

PCAP ANALYSIS REPORT
Total Packets: 100
TCP Packets  : 100
UDP Packets  : 0
ICMP Packets : 0


Perfect. Since you already have:

```text
capture_20260712_123142.pcap
tcp_capture.pcap
```

Let's build a **real packet analysis script** that counts:

* TCP Packets
* UDP Packets
* ICMP Packets
* Other Packets

and prints percentages.

***

# Version 1: Basic TCP/UDP/ICMP Counter

```python
from scapy.all import rdpcap
from scapy.layers.inet import TCP, UDP, ICMP

packets = rdpcap(
    "tcp_capture.pcap"
)

tcp_count = 0
udp_count = 0
icmp_count = 0
other_count = 0

for packet in packets:

    if TCP in packet:

        tcp_count += 1

    elif UDP in packet:

        udp_count += 1

    elif ICMP in packet:

        icmp_count += 1

    else:

        other_count += 1

print("=" * 40)
print("PCAP ANALYSIS REPORT")
print("=" * 40)

print(
    "Total Packets:",
    len(packets)
)

print(
    "TCP Packets:",
    tcp_count
)

print(
    "UDP Packets:",
    udp_count
)

print(
    "ICMP Packets:",
    icmp_count
)

print(
    "Other Packets:",
    other_count
)
```

***

# Example Output

```text
========================================
PCAP ANALYSIS REPORT
========================================

Total Packets: 150

TCP Packets: 110
UDP Packets: 30
ICMP Packets: 5
Other Packets: 5
```

***

# Version 2: Add Percentages

```python
from scapy.all import rdpcap
from scapy.layers.inet import TCP, UDP, ICMP

packets = rdpcap(
    "tcp_capture.pcap"
)

tcp_count = 0
udp_count = 0
icmp_count = 0
other_count = 0

for packet in packets:

    if TCP in packet:

        tcp_count += 1

    elif UDP in packet:

        udp_count += 1

    elif ICMP in packet:

        icmp_count += 1

    else:

        other_count += 1

total = len(packets)

print("=" * 50)
print("TRAFFIC BREAKDOWN")
print("=" * 50)

print(
    f"TCP  : {tcp_count} ({tcp_count/total*100:.2f}%)"
)

print(
    f"UDP  : {udp_count} ({udp_count/total*100:.2f}%)"
)

print(
    f"ICMP : {icmp_count} ({icmp_count/total*100:.2f}%)"
)

print(
    f"OTHER: {other_count} ({other_count/total*100:.2f}%)"
)
```

***

# Version 3: Show Top Destination Ports

Very useful for network engineers.

```python
from scapy.all import rdpcap
from scapy.layers.inet import TCP

packets = rdpcap(
    "tcp_capture.pcap"
)

ports = {}

for packet in packets:

    if TCP in packet:

        dport = packet[TCP].dport

        ports[dport] = (
            ports.get(dport, 0)
            + 1
        )

print("=" * 40)
print("TOP TCP PORTS")
print("=" * 40)

for port, count in sorted(
    ports.items(),
    key=lambda x: x[1],
    reverse=True
):

    print(
        f"Port {port}: {count}"
    )
```

Example:

```text
Port 443: 80
Port 80: 15
Port 22: 5
```

***

# Version 4: Find Top Talkers (Most Active IPs)

```python
from scapy.all import rdpcap
from scapy.layers.inet import IP

packets = rdpcap(
    "tcp_capture.pcap"
)

hosts = {}

for packet in packets:

    if IP in packet:

        src = packet[IP].src

        hosts[src] = (
            hosts.get(src, 0)
            + 1
        )

print("=" * 40)
print("TOP TALKERS")
print("=" * 40)

for ip, count in sorted(
    hosts.items(),
    key=lambda x: x[1],
    reverse=True
):

    print(
        ip,
        count
    )
```

***

# Version 5: Production Style Summary

```python
from scapy.all import rdpcap
from scapy.layers.inet import TCP, UDP, ICMP, IP

packets = rdpcap(
    "tcp_capture.pcap"
)

tcp = 0
udp = 0
icmp = 0

unique_hosts = set()

for packet in packets:

    if IP in packet:

        unique_hosts.add(
            packet[IP].src
        )

    if TCP in packet:

        tcp += 1

    elif UDP in packet:

        udp += 1

    elif ICMP in packet:

        icmp += 1

print("\n")
print("=" * 50)
print("NETWORK TRAFFIC REPORT")
print("=" * 50)

print(
    "Total Packets :",
    len(packets)
)

print(
    "TCP Packets   :",
    tcp
)

print(
    "UDP Packets   :",
    udp
)

print(
    "ICMP Packets  :",
    icmp
)

print(
    "Unique Hosts  :",
    len(unique_hosts)
)
```

Example Output:

```text
==================================================
NETWORK TRAFFIC REPORT
==================================================

Total Packets : 500
TCP Packets   : 420
UDP Packets   : 60
ICMP Packets  : 15
Unique Hosts  : 12
```

***

### Next Production-Level Exercises

1. Detect DNS packets (`UDP/53`)
2. Count HTTPS traffic (`TCP/443`)
3. Calculate top source IPs
4. Calculate top destination IPs
5. Detect ICMP echo requests/replies
6. Generate CSV reports from PCAPs
7. Detect TCP SYN packets
8. Detect retransmissions using Wireshark filters after capture

These are the same types of analyses used during WAN, Azure, and network troubleshooting investigations.


In [19]:
from scapy.all import rdpcap
from scapy.layers.inet import TCP, UDP, ICMP

packets = rdpcap(
    "tcp_capture.pcap"
)

tcp_count = 0
udp_count = 0
icmp_count = 0
other_count = 0

for packet in packets:

    if TCP in packet:

        tcp_count += 1

    elif UDP in packet:

        udp_count += 1

    elif ICMP in packet:

        icmp_count += 1

    else:

        other_count += 1

print("=" * 40)
print("PCAP ANALYSIS REPORT")
print("=" * 40)

print(
    "Total Packets:",
    len(packets)
)

print(
    "TCP Packets:",
    tcp_count
)

print(
    "UDP Packets:",
    udp_count
)

print(
    "ICMP Packets:",
    icmp_count
)

print(
    "Other Packets:",
    other_count
)

PCAP ANALYSIS REPORT
Total Packets: 100
TCP Packets: 100
UDP Packets: 0
ICMP Packets: 0
Other Packets: 0


In [20]:
from scapy.all import rdpcap
from scapy.layers.inet import TCP, UDP, ICMP

packets = rdpcap(
    "tcp_capture.pcap"
)

tcp_count = 0
udp_count = 0
icmp_count = 0
other_count = 0

for packet in packets:

    if TCP in packet:

        tcp_count += 1

    elif UDP in packet:

        udp_count += 1

    elif ICMP in packet:

        icmp_count += 1

    else:

        other_count += 1

total = len(packets)

print("=" * 50)
print("TRAFFIC BREAKDOWN")
print("=" * 50)

print(
    f"TCP  : {tcp_count} ({tcp_count/total*100:.2f}%)"
)

print(
    f"UDP  : {udp_count} ({udp_count/total*100:.2f}%)"
)

print(
    f"ICMP : {icmp_count} ({icmp_count/total*100:.2f}%)"
)

print(
    f"OTHER: {other_count} ({other_count/total*100:.2f}%)"
)

TRAFFIC BREAKDOWN
TCP  : 100 (100.00%)
UDP  : 0 (0.00%)
ICMP : 0 (0.00%)
OTHER: 0 (0.00%)


In [21]:
from scapy.all import rdpcap
from scapy.layers.inet import TCP

packets = rdpcap(
    "tcp_capture.pcap"
)

ports = {}

for packet in packets:

    if TCP in packet:

        dport = packet[TCP].dport

        ports[dport] = (
            ports.get(dport, 0)
            + 1
        )

print("=" * 40)
print("TOP TCP PORTS")
print("=" * 40)

for port, count in sorted(
    ports.items(),
    key=lambda x: x[1],
    reverse=True
):

    print(
        f"Port {port}: {count}"
    )

TOP TCP PORTS
Port 443: 49
Port 62581: 6
Port 62583: 6
Port 80: 6
Port 57629: 6
Port 52840: 3
Port 52838: 2
Port 52839: 2
Port 52841: 2
Port 52842: 2
Port 52843: 2
Port 53777: 1
Port 59316: 1
Port 65441: 1
Port 53904: 1
Port 5228: 1
Port 58929: 1
Port 49200: 1
Port 54560: 1
Port 50160: 1
Port 56619: 1
Port 58858: 1
Port 57255: 1
Port 59587: 1
Port 59450: 1


In [22]:
from scapy.all import rdpcap
from scapy.layers.inet import IP

packets = rdpcap(
    "tcp_capture.pcap"
)

hosts = {}

for packet in packets:

    if IP in packet:

        src = packet[IP].src

        hosts[src] = (
            hosts.get(src, 0)
            + 1
        )

print("=" * 40)
print("TOP TALKERS")
print("=" * 40)

for ip, count in sorted(
    hosts.items(),
    key=lambda x: x[1],
    reverse=True
):

    print(
        ip,
        count
    )

TOP TALKERS
10.0.9.200 56
40.64.145.160 12
142.250.73.78 6
40.74.98.198 6
8.8.8.8 4
168.63.129.16 3
52.108.8.12 1
52.112.95.40 1
13.66.138.105 1
52.108.9.12 1
74.125.199.188 1
52.107.248.5 1
142.250.9.94 1
35.186.224.24 1
52.110.2.9 1
52.107.248.21 1
52.123.129.14 1
20.42.65.90 1
20.59.87.225 1


In [23]:
from scapy.all import rdpcap
from scapy.layers.inet import TCP, UDP, ICMP, IP

packets = rdpcap(
    "tcp_capture.pcap"
)

tcp = 0
udp = 0
icmp = 0

unique_hosts = set()

for packet in packets:

    if IP in packet:

        unique_hosts.add(
            packet[IP].src
        )

    if TCP in packet:

        tcp += 1

    elif UDP in packet:

        udp += 1

    elif ICMP in packet:

        icmp += 1

print("\n")
print("=" * 50)
print("NETWORK TRAFFIC REPORT")
print("=" * 50)

print(
    "Total Packets :",
    len(packets)
)

print(
    "TCP Packets   :",
    tcp
)

print(
    "UDP Packets   :",
    udp
)

print(
    "ICMP Packets  :",
    icmp
)

print(
    "Unique Hosts  :",
    len(unique_hosts)
)



NETWORK TRAFFIC REPORT
Total Packets : 100
TCP Packets   : 100
UDP Packets   : 0
ICMP Packets  : 0
Unique Hosts  : 19


Here's a practical Scapy example to count **HTTPS traffic (TCP port 443)** from your existing PCAP file.

***

# Example 1: Count HTTPS Packets

```python
from scapy.all import rdpcap
from scapy.layers.inet import TCP

packets = rdpcap(
    "tcp_capture.pcap"
)

https_count = 0

for packet in packets:

    if TCP in packet:

        if (
            packet[TCP].sport == 443
            or
            packet[TCP].dport == 443
        ):

            https_count += 1

print(
    "Total Packets:",
    len(packets)
)

print(
    "HTTPS Packets:",
    https_count
)
```

### Example Output

```text
Total Packets: 500

HTTPS Packets: 420
```

***

# Example 2: Show HTTPS Packet Summaries

```python
from scapy.all import rdpcap
from scapy.layers.inet import TCP

packets = rdpcap(
    "tcp_capture.pcap"
)

for packet in packets:

    if TCP in packet:

        if (
            packet[TCP].sport == 443
            or
            packet[TCP].dport == 443
        ):

            print(
                packet.summary()
            )
```

Example:

```text
Ether / IP / TCP
192.168.1.10:55231 > 13.107.246.40:443

Ether / IP / TCP
13.107.246.40:443 > 192.168.1.10:55231
```

***

# Example 3: Count HTTP vs HTTPS

```python
from scapy.all import rdpcap
from scapy.layers.inet import TCP

packets = rdpcap(
    "tcp_capture.pcap"
)

http_count = 0
https_count = 0

for packet in packets:

    if TCP in packet:

        if (
            packet[TCP].sport == 80
            or
            packet[TCP].dport == 80
        ):

            http_count += 1

        elif (
            packet[TCP].sport == 443
            or
            packet[TCP].dport == 443
        ):

            https_count += 1

print(
    "HTTP Packets:",
    http_count
)

print(
    "HTTPS Packets:",
    https_count
)
```

***

# Example 4: Calculate HTTPS Percentage

```python
from scapy.all import rdpcap
from scapy.layers.inet import TCP

packets = rdpcap(
    "tcp_capture.pcap"
)

tcp_count = 0
https_count = 0

for packet in packets:

    if TCP in packet:

        tcp_count += 1

        if (
            packet[TCP].sport == 443
            or
            packet[TCP].dport == 443
        ):

            https_count += 1

print(
    "TCP Packets:",
    tcp_count
)

print(
    "HTTPS Packets:",
    https_count
)

print(
    f"HTTPS Percentage: {(https_count/tcp_count)*100:.2f}%"
)
```

Example:

```text
TCP Packets: 450
HTTPS Packets: 380
HTTPS Percentage: 84.44%
```

***

# Example 5: Production-Style HTTPS Report

```python
from scapy.all import rdpcap
from scapy.layers.inet import TCP, IP

packets = rdpcap(
    "tcp_capture.pcap"
)

https_count = 0
servers = {}

for packet in packets:

    if TCP in packet and IP in packet:

        if (
            packet[TCP].sport == 443
            or
            packet[TCP].dport == 443
        ):

            https_count += 1

            dst = packet[IP].dst

            servers[dst] = (
                servers.get(dst, 0)
                + 1
            )

print("=" * 50)
print("HTTPS TRAFFIC REPORT")
print("=" * 50)

print(
    "HTTPS Packets:",
    https_count
)

print("\nTOP HTTPS DESTINATIONS")

for ip, count in sorted(
    servers.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]:

    print(
        f"{ip} : {count}"
    )
```

Example:

```text
==================================================
HTTPS TRAFFIC REPORT
==================================================

HTTPS Packets: 380

TOP HTTPS DESTINATIONS

13.107.246.40 : 120
20.190.160.10 : 90
8.8.8.8 : 25
```

***

# Bonus: Count DNS + HTTP + HTTPS + ICMP Together

```python
from scapy.all import rdpcap
from scapy.layers.inet import TCP, UDP, ICMP

packets = rdpcap(
    "tcp_capture.pcap"
)

dns = 0
http = 0
https = 0
icmp = 0

for packet in packets:

    if ICMP in packet:
        icmp += 1

    if TCP in packet:

        if (
            packet[TCP].sport == 80
            or
            packet[TCP].dport == 80
        ):

            http += 1

        elif (
            packet[TCP].sport == 443
            or
            packet[TCP].dport == 443
        ):

            https += 1

    if UDP in packet:

        if (
            packet[UDP].sport == 53
            or
            packet[UDP].dport == 53
        ):

            dns += 1

print(
    "DNS  :", dns
)

print(
    "HTTP :", http
)

print(
    "HTTPS:", https
)

print(
    "ICMP :", icmp
)
```

This is very close to a real network-troubleshooting script used to quickly summarize traffic types from a capture file before opening it in Wireshark.


In [24]:
from scapy.all import rdpcap
from scapy.layers.inet import TCP

packets = rdpcap(
    "tcp_capture.pcap"
)

https_count = 0

for packet in packets:

    if TCP in packet:

        if (
            packet[TCP].sport == 443
            or
            packet[TCP].dport == 443
        ):

            https_count += 1

print(
    "Total Packets:",
    len(packets)
)

print(
    "HTTPS Packets:",
    https_count
)

Total Packets: 100
HTTPS Packets: 89


In [25]:
from scapy.all import rdpcap
from scapy.layers.inet import TCP

packets = rdpcap(
    "tcp_capture.pcap"
)

for packet in packets:

    if TCP in packet:

        if (
            packet[TCP].sport == 443
            or
            packet[TCP].dport == 443
        ):

            print(
                packet.summary()
            )

Ether / IP / TCP 10.0.9.200:62581 > 40.64.145.160:https PA / Raw
Ether / IP / TCP 10.0.9.200:62583 > 40.64.145.160:https PA / Raw
Ether / IP / TCP 40.64.145.160:https > 10.0.9.200:62581 A / Padding
Ether / IP / TCP 40.64.145.160:https > 10.0.9.200:62583 A / Padding
Ether / IP / TCP 10.0.9.200:52838 > 142.250.73.78:https S
Ether / IP / TCP 142.250.73.78:https > 10.0.9.200:52838 SA
Ether / IP / TCP 10.0.9.200:52838 > 142.250.73.78:https A
Ether / IP / TCP 10.0.9.200:52838 > 142.250.73.78:https FA
Ether / IP / TCP 142.250.73.78:https > 10.0.9.200:52838 FA / Padding
Ether / IP / TCP 10.0.9.200:52838 > 142.250.73.78:https A
Ether / IP / TCP 52.108.8.12:https > 10.0.9.200:53777 A / Padding
Ether / IP / TCP 10.0.9.200:53777 > 52.108.8.12:https A
Ether / IP / TCP 40.64.145.160:https > 10.0.9.200:62583 PA / Raw
Ether / IP / TCP 40.64.145.160:https > 10.0.9.200:62581 PA / Raw
Ether / IP / TCP 10.0.9.200:62581 > 40.64.145.160:https A
Ether / IP / TCP 10.0.9.200:62583 > 40.64.145.160:https A
Ether

In [26]:
from scapy.all import rdpcap
from scapy.layers.inet import TCP

packets = rdpcap(
    "tcp_capture.pcap"
)

http_count = 0
https_count = 0

for packet in packets:

    if TCP in packet:

        if (
            packet[TCP].sport == 80
            or
            packet[TCP].dport == 80
        ):

            http_count += 1

        elif (
            packet[TCP].sport == 443
            or
            packet[TCP].dport == 443
        ):

            https_count += 1

print(
    "HTTP Packets:",
    http_count
)

print(
    "HTTPS Packets:",
    https_count
)

HTTP Packets: 9
HTTPS Packets: 89


In [27]:
from scapy.all import rdpcap
from scapy.layers.inet import TCP

packets = rdpcap(
    "tcp_capture.pcap"
)

tcp_count = 0
https_count = 0

for packet in packets:

    if TCP in packet:

        tcp_count += 1

        if (
            packet[TCP].sport == 443
            or
            packet[TCP].dport == 443
        ):

            https_count += 1

print(
    "TCP Packets:",
    tcp_count
)

print(
    "HTTPS Packets:",
    https_count
)

print(
    f"HTTPS Percentage: {(https_count/tcp_count)*100:.2f}%"
)

TCP Packets: 100
HTTPS Packets: 89
HTTPS Percentage: 89.00%


In [28]:
from scapy.all import rdpcap
from scapy.layers.inet import TCP, IP

packets = rdpcap(
    "tcp_capture.pcap"
)

https_count = 0
servers = {}

for packet in packets:

    if TCP in packet and IP in packet:

        if (
            packet[TCP].sport == 443
            or
            packet[TCP].dport == 443
        ):

            https_count += 1

            dst = packet[IP].dst

            servers[dst] = (
                servers.get(dst, 0)
                + 1
            )

print("=" * 50)
print("HTTPS TRAFFIC REPORT")
print("=" * 50)

print(
    "HTTPS Packets:",
    https_count
)

print("\nTOP HTTPS DESTINATIONS")

for ip, count in sorted(
    servers.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]:

    print(
        f"{ip} : {count}"
    )

HTTPS TRAFFIC REPORT
HTTPS Packets: 89

TOP HTTPS DESTINATIONS
10.0.9.200 : 40
40.64.145.160 : 12
142.250.73.78 : 12
8.8.8.8 : 9
40.74.98.198 : 3
13.66.138.105 : 2
52.108.8.12 : 1
52.112.95.40 : 1
52.108.9.12 : 1
52.107.248.5 : 1


In [29]:
from scapy.all import rdpcap
from scapy.layers.inet import TCP, UDP, ICMP

packets = rdpcap(
    "tcp_capture.pcap"
)

dns = 0
http = 0
https = 0
icmp = 0

for packet in packets:

    if ICMP in packet:
        icmp += 1

    if TCP in packet:

        if (
            packet[TCP].sport == 80
            or
            packet[TCP].dport == 80
        ):

            http += 1

        elif (
            packet[TCP].sport == 443
            or
            packet[TCP].dport == 443
        ):

            https += 1

    if UDP in packet:

        if (
            packet[UDP].sport == 53
            or
            packet[UDP].dport == 53
        ):

            dns += 1

print(
    "DNS  :", dns
)

print(
    "HTTP :", http
)

print(
    "HTTPS:", https
)

print(
    "ICMP :", icmp
)

DNS  : 0
HTTP : 9
HTTPS: 89
ICMP : 0
